# 02. Semantic Understanding & Multilingual

Covers **Attributes 6 & 7**:
- Entity aliases (Bud, BL, Stella, Ultra, Brahma)
- Geographic abbreviations (US, UK, America, Britain)
- Typo tolerance (coron, hoegarden)
- Multilingual queries (Spanish, French, Hindi)
- Mixed-language queries (Spanglish, Hinglish)

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Brand Aliases, Abbreviations & Typo Correction

In [2]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
queries = [
    "What was Bud revenue in US in 2025?",
    "Show BL volume in US in 2025",
    "Show Ultra market share in America in 2025",
    "What was Coron sales in Mexico in 2025?",
    "Hoegarden volume in China in 2025"
]
for q in queries:
    r = orch.handle_turn(q)
    print(f"Query: {q}\nResolved SQL: {r.sql_used}\n")

Query: What was Bud revenue in US in 2025?
Resolved SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Budweiser' AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500

Query: Show BL volume in US in 2025
Resolved SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Bud Light' AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500

Query: Show Ultra market share in America in 2025
Resolved SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Michelob ULTRA' AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500

Query: What was Coron sales in M

### 2. Multilingual & Mixed-Language Support

In [3]:
multilingual_queries = [
    "¿Cuáles fueron los ingresos de Corona en México en 2025?",
    "Quelle était la part de marché de Stella Artois en Belgique en 2024?",
    "2025 mein Budweiser ka revenue United States mein kitna tha?",
    "Show me los ingresos de Bud Light in US para 2025"
]
for q in multilingual_queries:
    r = orch.handle_turn(q)
    lang = r.raw_nlu.get("language", "en")
    print(f"Query: {q}\nDetected Language: {lang}\nAnswer: {r.answer[:140]}...\n")

Query: ¿Cuáles fueron los ingresos de Corona en México en 2025?
Detected Language: es
Answer: En 2025, Corona in Mexico registró ingresos netos de **$11,209,049** y un volumen de **113,001.2 hL**.

| Brand | Country | Year | Net Reven...

Query: Quelle était la part de marché de Stella Artois en Belgique en 2024?
Detected Language: fr
Answer: Les données de performance d'AB InBev pour la période demandée sont présentées ci-dessous:

| Brand | Country | Year | Net Revenue (USD) | V...

Query: 2025 mein Budweiser ka revenue United States mein kitna tha?
Detected Language: hi
Answer: AB InBev ke reporting ke anusaar, brand ka performance data neeche table mein darshaya gaya hai:

| Brand | Country | Year | Net Revenue (US...

Query: Show me los ingresos de Bud Light in US para 2025
Detected Language: es
Answer: En 2025, Bud Light in United States registró ingresos netos de **$4,623,330** y un volumen de **78,096.7 hL**.

| Brand | Country | Year | N...
